# Model Monitoring: Drift Detection & Alerting

Once a model is in production, its performance can degrade silently. This notebook covers:
1. **Data drift** detection (PSI, KS test)
2. **Model performance** monitoring over time
3. **Alerting logic** and thresholds
4. **Dashboard-style** visualisation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

%matplotlib inline
np.random.seed(42)
print('Setup complete.')

## 1. Simulate a Production Scenario

We train on "historical" data, then simulate production batches where the feature
distribution gradually shifts (covariate shift).

In [ ]:
# Train a model on the reference data
data = load_breast_cancer()
X_ref, X_hold, y_ref, y_hold = train_test_split(
    data.data, data.target, test_size=0.3, random_state=42
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_ref, y_ref)
print(f'Reference accuracy: {accuracy_score(y_hold, model.predict(X_hold)):.4f}')

# Simulate production batches with increasing drift
n_batches = 8
batch_size = 60
production_batches = []
for i in range(n_batches):
    shift = i * 0.3  # increasing shift magnitude
    X_batch = X_hold[np.random.choice(len(X_hold), batch_size)] + np.random.normal(shift, 0.5 + i * 0.1, (batch_size, X_ref.shape[1]))
    y_batch = y_hold[np.random.choice(len(y_hold), batch_size)]
    production_batches.append((X_batch, y_batch))

print(f'Simulated {n_batches} production batches of size {batch_size}')

## 2. Population Stability Index (PSI)

PSI measures how much a feature distribution has shifted from the reference.

$$\text{PSI} = \sum_{i=1}^{B} (p_i - q_i) \ln\left(\frac{p_i}{q_i}\right)$$

- PSI < 0.1: no significant shift
- 0.1 <= PSI < 0.25: moderate shift
- PSI >= 0.25: significant shift

In [ ]:
def compute_psi(reference, current, n_bins=10):
    """Compute PSI between reference and current distributions."""
    eps = 1e-6
    # Use reference quantiles for binning
    breakpoints = np.quantile(reference, np.linspace(0, 1, n_bins + 1))
    breakpoints[0] = -np.inf
    breakpoints[-1] = np.inf
    
    ref_counts = np.histogram(reference, bins=breakpoints)[0] / len(reference)
    cur_counts = np.histogram(current, bins=breakpoints)[0] / len(current)
    
    ref_counts = np.clip(ref_counts, eps, None)
    cur_counts = np.clip(cur_counts, eps, None)
    
    psi = np.sum((cur_counts - ref_counts) * np.log(cur_counts / ref_counts))
    return psi

# Compute PSI for each batch, feature 0
feature_idx = 0
psi_values = []
for X_batch, _ in production_batches:
    psi = compute_psi(X_ref[:, feature_idx], X_batch[:, feature_idx])
    psi_values.append(psi)

print('PSI per batch (feature 0):')
for i, psi in enumerate(psi_values):
    status = 'OK' if psi < 0.1 else ('WARN' if psi < 0.25 else 'ALERT')
    print(f'  Batch {i+1}: PSI = {psi:.4f} [{status}]')

## 3. Kolmogorov-Smirnov Test for Drift

The KS test is a non-parametric test comparing two distributions.
A low p-value (< 0.05) indicates the distributions differ significantly.

In [ ]:
ks_results = []
for i, (X_batch, _) in enumerate(production_batches):
    # Test top 5 features
    batch_ks = {}
    for feat_idx in range(5):
        stat, pval = stats.ks_2samp(X_ref[:, feat_idx], X_batch[:, feat_idx])
        batch_ks[data.feature_names[feat_idx]] = {'statistic': stat, 'p_value': pval}
    ks_results.append(batch_ks)

# Display results for first and last batch
for label, idx in [('Batch 1 (low drift)', 0), ('Batch 8 (high drift)', -1)]:
    print(f'\n{label}:')
    for feat, res in ks_results[idx].items():
        drift = 'DRIFT' if res['p_value'] < 0.05 else 'OK'
        print(f'  {feat:30s} KS={res["statistic"]:.4f}  p={res["p_value"]:.4f}  [{drift}]')

In [ ]:
# Track metrics over batches
metrics_log = []
for i, (X_batch, y_batch) in enumerate(production_batches):
    y_pred = model.predict(X_batch)
    y_proba = model.predict_proba(X_batch)[:, 1]
    metrics_log.append({
        'batch': i + 1,
        'accuracy': accuracy_score(y_batch, y_pred),
        'f1': f1_score(y_batch, y_pred, zero_division=0),
        'auc': roc_auc_score(y_batch, y_proba) if len(np.unique(y_batch)) > 1 else np.nan,
        'psi_feat0': psi_values[i],
    })

df_metrics = pd.DataFrame(metrics_log)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, color in zip(axes, ['accuracy', 'f1', 'psi_feat0'], ['steelblue', 'darkorange', 'crimson']):
    ax.plot(df_metrics['batch'], df_metrics[col], 'o-', color=color)
    ax.set_xlabel('Batch')
    ax.set_ylabel(col)
    ax.set_title(col.upper())
    if col == 'psi_feat0':
        ax.axhline(0.1, ls='--', color='orange', label='Warn (0.1)')
        ax.axhline(0.25, ls='--', color='red', label='Alert (0.25)')
        ax.legend()
plt.tight_layout()
plt.show()

# --- 4. Model Performance Monitoring ---
# Track metrics over batches
metrics_log = []
for i, (X_batch, y_batch) in enumerate(production_batches):
    y_pred = model.predict(X_batch)
    y_proba = model.predict_proba(X_batch)[:, 1]
    metrics_log.append({
        'batch': i + 1,
        'accuracy': accuracy_score(y_batch, y_pred),
        'f1': f1_score(y_batch, y_pred, zero_division=0),
        'auc': roc_auc_score(y_batch, y_proba) if len(np.unique(y_batch)) > 1 else np.nan,
        'psi_feat0': psi_values[i],
    })

df_metrics = pd.DataFrame(metrics_log)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, color in zip(axes, ['accuracy', 'f1', 'psi_feat0'], ['steelblue', 'darkorange', 'crimson']):
    ax.plot(df_metrics['batch'], df_metrics[col], 'o-', color=color)
    ax.set_xlabel('Batch')
    ax.set_ylabel(col)
    ax.set_title(col.upper())
    if col == 'psi_feat0':
        ax.axhline(0.1, ls='--', color='orange', label='Warn (0.1)')
        ax.axhline(0.25, ls='--', color='red', label='Alert (0.25)')
        ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
class ModelMonitor:
    def __init__(self, accuracy_threshold=0.85, psi_warn=0.1, psi_alert=0.25):
        self.accuracy_threshold = accuracy_threshold
        self.psi_warn = psi_warn
        self.psi_alert = psi_alert
        self.alerts = []

    def check(self, batch_id, accuracy, psi):
        alerts = []
        if accuracy < self.accuracy_threshold:
            alerts.append(f'[PERF] Batch {batch_id}: accuracy={accuracy:.3f} < {self.accuracy_threshold}')
        if psi >= self.psi_alert:
            alerts.append(f'[DRIFT-ALERT] Batch {batch_id}: PSI={psi:.4f} >= {self.psi_alert}')
        elif psi >= self.psi_warn:
            alerts.append(f'[DRIFT-WARN] Batch {batch_id}: PSI={psi:.4f} >= {self.psi_warn}')
        self.alerts.extend(alerts)
        return alerts

monitor = ModelMonitor(accuracy_threshold=0.85, psi_warn=0.1, psi_alert=0.25)

for _, row in df_metrics.iterrows():
    alerts = monitor.check(int(row['batch']), row['accuracy'], row['psi_feat0'])
    for a in alerts:
        print(a)

print(f'\nTotal alerts raised: {len(monitor.alerts)}')

## Key Takeaways

- **Data drift** (covariate shift) degrades model performance silently -- monitor it.
- **PSI** and **KS test** are complementary drift detection methods.
- Track **business and ML metrics** per batch or time window.
- Build **automated alerting** with clear thresholds and escalation.
- Tools: Evidently AI, WhyLabs, NannyML, custom Prometheus + Grafana.

**Workflow:** Detect drift -> Alert -> Investigate -> Retrain -> Redeploy.